# 🔮 Daily Grand Lotto: Multi-Scale Predictor (RUNNER)
- **Primary Prediction Engine**: Uses a high-complexity Stacking Ensemble (Random Forest, XGBoost, and Gradient Boosting) to generate optimized numbers for the next draw.
- **Advanced Context**: Analyzes short-term sequences (last 3 draws), mid-term trends (last 10 draws), and long-term history (all-time frequency and gap analysis) to find statistical edges.

In [2]:
# === Daily Grand Lotto Multi-Scale Predictor (HONEST-AUDIT RUNNER) ===
# Same model pipeline as before, but the audit now scores the model AGAINST
# a random-pick baseline, so the scorecard reflects reality instead of noise.

import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta
from pathlib import Path
from math import comb
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from collections import Counter
from IPython.display import display, HTML

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

RNG = np.random.default_rng(2026)
N_BASELINE_TICKETS = 1000  # random tickets simulated per audited draw

# Theoretical expectation: matches for ANY fixed 5-of-49 pick
CHANCE_EXPECTATION = sum(k * comb(5, k) * comb(44, 5 - k) / comb(49, 5) for k in range(6))  # = 0.5102

# --- 1. Load Data ---
csv_path = "daily_grand_results.csv"
df = pd.read_csv(csv_path).sort_values(by=["year", "month", "day"]).reset_index(drop=True)
number_cols = ['dn_1', 'dn_2', 'dn_3', 'dn_4', 'dn_5', 'grand_number']
main_cols = ['dn_1', 'dn_2', 'dn_3', 'dn_4', 'dn_5']

# --- 2. Feature Engineering (unchanged) ---
def get_gap_stats(df, current_idx):
    history = df.iloc[:current_idx][main_cols].values
    gaps = {}
    for n in range(1, 50):
        found = False
        for i, draw in enumerate(reversed(history)):
            if n in draw:
                gaps[n] = i; found = True; break
        if not found:
            gaps[n] = len(history)
    return gaps

def add_multi_scale_features(df, current_idx, window_small=3, window_med=22):
    features = []
    past_small = df.iloc[current_idx - window_small: current_idx][number_cols].values.flatten()
    features.extend(past_small)
    past_med = df.iloc[current_idx - window_med: current_idx][main_cols]
    features.append(past_med.values.mean())
    features.append(past_med.values.std())
    recent_gaps = get_gap_stats(df, current_idx)
    old_gaps = get_gap_stats(df, max(0, current_idx - 10))
    features.append(np.mean(list(recent_gaps.values())) - np.mean(list(old_gaps.values())))
    history_recent = df.iloc[max(0, current_idx - 50):current_idx][main_cols].values.flatten()
    freq_map = Counter(history_recent)
    for num in df.iloc[current_idx - 1][main_cols]:
        features.append(freq_map.get(num, 0))
    return np.array(features)

# --- 3. Build Training Set ---
X, y = [], []
for i in range(22, len(df)):
    X.append(add_multi_scale_features(df, i))
    y.append(df.iloc[i][number_cols].values)
X, y = np.array(X), np.array(y)

# --- 4. Ensemble ---
estimators = [
    ('gb', GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.07, random_state=42)),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)),
]
if HAS_XGB:
    estimators.append(('xgb', XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.07, verbosity=0)))

model = MultiOutputRegressor(StackingRegressor(
    estimators=estimators, final_estimator=RidgeCV(), n_jobs=-1
))
model.fit(X, y)

# --- 5. Prediction Logic ---
def refine_prediction(pred):
    main = []
    for val in np.sort(pred[:5]):
        val = int(round(val))
        while val in main or val < 1 or val > 49:
            val = val + 1 if val < 49 else val - 1
        main.append(val)
    grand = int(max(1, min(7, round(pred[5]))))
    return sorted(main) + [grand]

next_feat = add_multi_scale_features(df, len(df)).reshape(1, -1)
res = refine_prediction(model.predict(next_feat)[0])

last_row = df.iloc[-1]
last_date = datetime(int(last_row['year']), int(last_row['month']), int(last_row['day']))
target_date = last_date + timedelta(days=(3 if last_date.weekday() < 3 else 4))

print("\n" + "═" * 50)
print(f" 🔮  SYSTEM PREDICTION: {target_date.strftime('%Y-%m-%d')}")
print(f" 🔢  Numbers: {res[:5]}   ⭐ Grand: {res[5]}")
print("═" * 50)

# --- 6. Save Prediction ---
pred_path = Path("future_draws.csv")
new_row = pd.DataFrame([{
    "Next Draw Date": target_date.strftime("%Y-%m-%d"),
    "dn_1": res[0], "dn_2": res[1], "dn_3": res[2],
    "dn_4": res[3], "dn_5": res[4], "grand_number": res[5]
}])

if pred_path.exists():
    existing = pd.read_csv(pred_path)
    existing["Next Draw Date"] = existing["Next Draw Date"].astype(str)
    if target_date.strftime("%Y-%m-%d") not in existing["Next Draw Date"].values:
        pd.concat([new_row, existing]).to_csv(pred_path, index=False)
        print(f"\n💾 Prediction for {target_date.strftime('%Y-%m-%d')} saved.")
    else:
        print(f"\nℹ️ Prediction for {target_date.strftime('%Y-%m-%d')} already exists. Skipping save.")
else:
    new_row.to_csv(pred_path, index=False)
    print("\n💾 Created future_draws.csv and saved first prediction.")

# --- 7. HONEST Audit: Model vs Random Baseline vs Theory ---
print("\n" + "─" * 50)
print(" 🔍  AUDIT: MODEL vs RANDOM BASELINE")
print("─" * 50)

def style_line(nums, highlight_set, color):
    return " ".join([
        f'<span style="background-color:{color}; color:white; padding:2px 6px; '
        f'border-radius:4px; font-weight:bold;">{n}</span>' if n in highlight_set
        else f'<span>{n}</span>' for n in nums
    ])

if pred_path.exists():
    actuals = pd.read_csv(csv_path)
    preds = pd.read_csv(pred_path)
    actuals['Date'] = pd.to_datetime(actuals[['year', 'month', 'day']]).dt.strftime('%Y-%m-%d')
    preds['Date'] = pd.to_datetime(preds['Next Draw Date']).dt.strftime('%Y-%m-%d')
    audit_merge = pd.merge(actuals, preds, on='Date', suffixes=('_act', '_pre'))

    audit_rows, model_scores, baseline_scores, grand_hits = [], [], [], 0

    for _, row in audit_merge.iterrows():
        act_main = [int(row[f'dn_{i}_act']) for i in range(1, 6)]
        pre_main = [int(row[f'dn_{i}_pre']) for i in range(1, 6)]
        act_g, pre_g = int(row['grand_number_act']), int(row['grand_number_pre'])

        matches = len(set(act_main) & set(pre_main))
        model_scores.append(matches)
        g_match = act_g == pre_g
        grand_hits += g_match

        # Random baseline: N simulated quick-picks scored against the SAME draw
        sims = np.array([
            len(set(RNG.choice(np.arange(1, 50), 5, replace=False)) & set(act_main))
            for _ in range(N_BASELINE_TICKETS)
        ])
        baseline_avg = sims.mean()
        baseline_scores.append(baseline_avg)
        # Where does the model's score rank among random tickets?
        beat_pct = (sims < matches).mean() * 100

        audit_rows.append({
            "Date": row['Date'],
            "Drawn": style_line(act_main, set(pre_main), "#2ca25f"),
            "Predicted": style_line(pre_main, set(act_main), "#3182bd"),
            "Model Hits": f"<b>{matches}/5</b>",
            "Random Baseline": f"{baseline_avg:.2f}/5",
            "Beats % of Random Tickets": f"{beat_pct:.0f}%",
            "Grand": f'{"✅" if g_match else "❌"} ({pre_g} vs {act_g})'
        })

    if audit_rows:
        audit_df = pd.DataFrame(audit_rows).sort_values(by="Date", ascending=False)
        display(HTML("<h3>📊 Model vs Random Baseline</h3>"))
        display(HTML(audit_df.to_html(escape=False, index=False)))

        n = len(model_scores)
        model_avg = np.mean(model_scores)
        base_avg = np.mean(baseline_scores)
        edge = model_avg - CHANCE_EXPECTATION

        # Hypergeometric std of matches for one draw ≈ 0.6577
        per_draw_std = 0.6577
        se = per_draw_std / np.sqrt(n)
        z = edge / se if se > 0 else 0.0

        print("\n" + "═" * 56)
        print(" 🏆  HONEST SCORECARD")
        print("─" * 56)
        print(f" Draws audited:              {n}")
        print(f" Model avg match:            {model_avg:.3f} / 5")
        print(f" Random-ticket avg match:    {base_avg:.3f} / 5")
        print(f" Theoretical chance:         {CHANCE_EXPECTATION:.3f} / 5")
        print(f" Model edge over chance:     {edge:+.3f}  (z = {z:+.2f})")
        print(f" Grand Number hits:          {grand_hits}/{n} (chance: {n/7:.1f})")
        print("─" * 56)
        if abs(z) < 2:
            verdict = ("🎲 INDISTINGUISHABLE FROM RANDOM — the model is performing\n"
                       "    exactly as chance predicts. This is the expected outcome\n"
                       "    for a fair lottery; |z| < 2 means no statistical edge.")
        elif z >= 2:
            verdict = (f"⚠️ ABOVE CHANCE at z = {z:.2f} — before celebrating, note that\n"
                       f"    with small samples this happens by luck ~2.5% of the time.\n"
                       f"    It will regress to 0.51 as more draws accumulate.")
        else:
            verdict = (f"📉 BELOW CHANCE at z = {z:.2f} — also just luck; random picks\n"
                       f"    underperform expectation about as often as they overperform.")
        print(f" Verdict:\n    {verdict}")
        print("═" * 56)
    else:
        print("ℹ️ No matching dates between results and future_draws.csv yet.")
else:
    print("⚠️ future_draws.csv not found.")

joblib.dump(model, 'daily_grand_ai_brain_tuned.pkl')


══════════════════════════════════════════════════
 🔮  SYSTEM PREDICTION: 2026-06-11
 🔢  Numbers: [8, 17, 25, 34, 42]   ⭐ Grand: 4
══════════════════════════════════════════════════

ℹ️ Prediction for 2026-06-11 already exists. Skipping save.

──────────────────────────────────────────────────
 🔍  AUDIT: MODEL vs RANDOM BASELINE
──────────────────────────────────────────────────


Date,Drawn,Predicted,Model Hits,Random Baseline,Beats % of Random Tickets,Grand
2026-05-18,6 7 28 36 48,8 16 25 32 42,0/5,0.49/5,0%,✅ (3 vs 3)
2026-05-04,1 16 21 32 41,8 16 25 33 42,1/5,0.51/5,56%,❌ (3 vs 4)
2026-04-13,2 11 19 29 30,7 16 24 33 42,0/5,0.52/5,0%,❌ (3 vs 4)
2026-03-23,8 17 28 37 46,7 17 25 33 42,1/5,0.49/5,58%,❌ (3 vs 7)
2026-03-02,5 15 28 31 34,8 16 25 33 42,0/5,0.55/5,0%,❌ (3 vs 7)
2026-02-05,4 22 25 33 49,8 16 24 33 42,1/5,0.46/5,60%,❌ (3 vs 1)
2026-02-02,5 14 40 43 49,8 16 25 33 42,0/5,0.51/5,0%,❌ (3 vs 1)
2026-01-29,6 12 21 37 39,7 16 25 34 42,0/5,0.49/5,0%,❌ (3 vs 4)
2026-01-26,4 6 28 45 48,8 16 25 33 42,0/5,0.50/5,0%,❌ (3 vs 2)
2026-01-19,10 17 25 32 41,8 17 24 33 42,1/5,0.54/5,54%,✅ (3 vs 3)



════════════════════════════════════════════════════════
 🏆  HONEST SCORECARD
────────────────────────────────────────────────────────
 Draws audited:              22
 Model avg match:            0.500 / 5
 Random-ticket avg match:    0.505 / 5
 Theoretical chance:         0.510 / 5
 Model edge over chance:     -0.010  (z = -0.07)
 Grand Number hits:          4/22 (chance: 3.1)
────────────────────────────────────────────────────────
 Verdict:
    🎲 INDISTINGUISHABLE FROM RANDOM — the model is performing
    exactly as chance predicts. This is the expected outcome
    for a fair lottery; |z| < 2 means no statistical edge.
════════════════════════════════════════════════════════


['daily_grand_ai_brain_tuned.pkl']